In [1]:
import os
import re
import unicodedata
import qrcode
import qrcode.image.svg
import pandas as pd

def slugify(texto: str) -> str:
    """
    Normaliza uma string, removendo acentos, caracteres especiais e espaços,
    tornando-a segura para ser usada como nome de arquivo.

    Args:
        texto (str): O texto original a ser convertido.

    Returns:
        str: O texto formatado como slug.
    """
    # Remove os acentos da string
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    # Converte tudo para minúsculas
    texto = texto.lower()
    # Substitui qualquer coisa que não seja letra minúscula ou número por hífen
    texto = re.sub(r'[^a-z0-9]+', '-', texto)
    # Remove hífens sobrando no começo e no final da string
    return texto.strip('-')

def gerar_qrcodes_svg(dados_qr: dict[str, str], diretorio_saida: str = "py") -> None:
    """
    Gera arquivos de QR Code em formato SVG a partir de um dicionário.

    Args:
        dados_qr (dict[str, str]): Dicionário onde a chave é a base do nome do arquivo e o valor é a URL.
        diretorio_saida (str): O caminho do diretório onde os SVGs serão salvos.
    """
    if not os.path.exists(diretorio_saida):
        os.makedirs(diretorio_saida)

    svg_factory = qrcode.image.svg.SvgImage

    for chave, url in dados_qr.items():
        qr = qrcode.QRCode(
            version=1,
            error_correction=qrcode.constants.ERROR_CORRECT_L,
            box_size=10,
            border=0,
        )

        qr.add_data(url)
        qr.make(fit=True)

        img = qr.make_image(image_factory=svg_factory)

        # Usa a função slugify na chave do dicionário para criar um nome de arquivo seguro
        nome_seguro = slugify(chave)
        nome_arquivo = f"{nome_seguro}.svg"
        caminho_completo = os.path.join(diretorio_saida, nome_arquivo)

        with open(caminho_completo, "wb") as f:
            img.save(f)

        print(f"✅ QR Code gerado com sucesso: {caminho_completo}")


In [2]:
# Exemplo de uso:
if __name__ == "__main__":
    # Dicionário onde as chaves contêm caracteres especiais e espaços
    urls_para_processar = {
        "Meu Portfólio Pessoal!": "https://www.google.com",
        "Repositório do GitHub": "https://github.com",
        "Página da Wikipédia - Python": "https://www.wikipedia.org"
    }
    
    gerar_qrcodes_svg(urls_para_processar)


✅ QR Code gerado com sucesso: py/meu-portfolio-pessoal.svg
✅ QR Code gerado com sucesso: py/repositorio-do-github.svg
✅ QR Code gerado com sucesso: py/pagina-da-wikipedia-python.svg


In [3]:
df = pd.read_excel("novo-qrcode.xlsx", engine="openpyxl")
df.tail()

,Título,URL
0,Escola do hip hop,https://www.gov.br/mec/pt-br/escola-do-hip-hop


In [4]:
dicionario = dict(zip(df['Título'], df['URL']))
gerar_qrcodes_svg(dicionario)

✅ QR Code gerado com sucesso: py/escola-do-hip-hop.svg
